# CarbonTatvaAI - Enterprise ESG Chatbot

Owner track: **ESG Chatbot for Enterprises - Snehashish + Pratyush**

This notebook builds a practical ESG chatbot that can answer enterprise questions such as:

- How do peers disclose Scope 3?
- Generate a board oversight disclosure.
- What does BRSR require for energy?
- Compare us against ITC and Tata Steel.
- Draft CBAM-aligned disclosures.

The notebook uses retrieval over uploaded reports/KPI files and calls an Ollama model when available. If Ollama is not available, it uses a clearly marked mock response so the demo still runs.


In [ ]:
# Cell 1: Install lightweight dependencies
import importlib.util
import subprocess
import sys

def pip_install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

requirements = {
    "pypdf": "pypdf",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "openpyxl": "openpyxl",
}

for import_name, package in requirements.items():
    if importlib.util.find_spec(import_name) is None:
        pip_install(package)

print("Dependencies ready.")


In [ ]:
# Cell 2: Configuration
import csv
import json
import os
import re
import textwrap
import urllib.error
import urllib.request
from pathlib import Path

import pandas as pd
from pypdf import PdfReader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

IS_KAGGLE = Path("/kaggle/input").exists()
WORK_DIR = Path("/kaggle/working/carbontatva_enterprise_chatbot") if IS_KAGGLE else Path.cwd() / "carbontatva_enterprise_chatbot"
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Add reports, BRSR files, KPI tables, or extracted ESG text here.
DOCUMENT_DIRS = []
if IS_KAGGLE:
    DOCUMENT_DIRS.append(Path("/kaggle/input"))
for candidate in [Path("company_reports"), Path("data/kpi_summary"), Path("demo_progress")]:
    if candidate.exists():
        DOCUMENT_DIRS.append(candidate)

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://localhost:11434/api/generate")
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "carbontatva")
TOP_K = 5

print("Work dir:", WORK_DIR)
print("Document dirs:", [str(path) for path in DOCUMENT_DIRS])
print("Ollama model:", OLLAMA_MODEL)
print("Ollama URL:", OLLAMA_URL)


In [ ]:
# Cell 3: Document loaders
SUPPORTED_SUFFIXES = {".pdf", ".txt", ".md", ".csv", ".json", ".jsonl", ".xlsx", ".xls"}
SKIP_PARTS = {"__pycache__", ".git", "checkpoint", "final_lora_adapter", "ollama_export"}

def clean_text(value):
    value = "" if value is None else str(value)
    value = value.replace("\ufeff", " ")
    value = re.sub(r"\s+", " ", value)
    return value.strip()

def should_skip(path):
    lowered = str(path).lower()
    return any(part in lowered for part in SKIP_PARTS)

def read_pdf(path):
    pages = []
    reader = PdfReader(str(path))
    for page_number, page in enumerate(reader.pages, 1):
        text = clean_text(page.extract_text() or "")
        if text:
            pages.append({
                "source": f"{path.name} page {page_number}",
                "text": text,
            })
    return pages

def read_table(path):
    suffix = path.suffix.lower()
    if suffix == ".csv":
        df = pd.read_csv(path, nrows=300)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path, nrows=300)
    elif suffix == ".jsonl":
        df = pd.read_json(path, lines=True).head(300)
    elif suffix == ".json":
        try:
            df = pd.read_json(path, lines=True).head(300)
        except ValueError:
            df = pd.read_json(path).head(300)
    else:
        raise ValueError(f"Unsupported table format: {path}")
    return [{
        "source": path.name,
        "text": df.fillna("").astype(str).to_csv(index=False),
    }]

def read_file(path):
    suffix = path.suffix.lower()
    if suffix == ".pdf":
        return read_pdf(path)
    if suffix in {".txt", ".md"}:
        return [{"source": path.name, "text": clean_text(path.read_text(encoding="utf-8", errors="ignore"))}]
    if suffix in {".csv", ".json", ".jsonl", ".xlsx", ".xls"}:
        return read_table(path)
    return []

def discover_files(dirs):
    files = []
    for folder in dirs:
        if not folder.exists():
            continue
        for path in folder.rglob("*"):
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES and not should_skip(path):
                files.append(path)
    return sorted(files)

files = discover_files(DOCUMENT_DIRS)
print(f"Found {len(files)} candidate files.")
for path in files[:20]:
    print("-", path)


In [ ]:
# Cell 4: Build retrieval corpus
DEMO_DOCUMENTS = [
    {
        "source": "demo_peer_scope3.txt",
        "text": (
            "Peer Scope 3 disclosures typically describe purchased goods and services, fuel and energy related activities, "
            "upstream transportation, business travel, employee commuting, downstream transportation, use of sold products, "
            "and end-of-life treatment where material. Strong disclosures state the boundary, method, emission factors, "
            "data quality limits, and categories excluded due to unavailable data."
        ),
    },
    {
        "source": "demo_brsr_energy.txt",
        "text": (
            "BRSR energy disclosures commonly require total energy consumption, energy intensity, renewable and non-renewable "
            "energy split where available, and year-on-year movement. Companies should avoid adding unsupported targets or "
            "initiatives when only KPI values are provided."
        ),
    },
    {
        "source": "demo_board_oversight.txt",
        "text": (
            "A board oversight disclosure should explain whether the board or a committee reviews ESG and climate matters, "
            "how frequently it receives updates, which executives are responsible, and how risks, targets, and compliance "
            "issues are escalated. If governance evidence is missing, the disclosure should say that it is not available."
        ),
    },
    {
        "source": "demo_cbam.txt",
        "text": (
            "CBAM-aligned disclosure for carbon-intensive exports should cover embedded emissions, product boundary, methodology, "
            "verification status, data gaps, supplier or facility level evidence, and transition actions. It should not claim "
            "CBAM compliance unless verification and reporting evidence is available."
        ),
    },
]

def chunk_document(doc, chunk_words=220, overlap=45):
    words = doc["text"].split()
    if len(words) <= chunk_words:
        return [{"source": doc["source"], "text": doc["text"]}]
    chunks = []
    step = max(1, chunk_words - overlap)
    for start in range(0, len(words), step):
        piece = " ".join(words[start : start + chunk_words]).strip()
        if len(piece) > 80:
            chunks.append({"source": doc["source"], "text": piece})
    return chunks

documents = []
errors = []
for path in files:
    try:
        documents.extend(read_file(path))
    except Exception as exc:
        errors.append(f"{path}: {exc}")

if not documents:
    print("No usable uploaded files found. Using demo ESG knowledge snippets.")
    documents = DEMO_DOCUMENTS

chunks = []
for document in documents:
    chunks.extend(chunk_document(document))

vectorizer = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2))
chunk_texts = [item["text"] for item in chunks]
matrix = vectorizer.fit_transform(chunk_texts)

print(f"Loaded {len(documents)} documents and built {len(chunks)} retrieval chunks.")
if errors:
    print("Some files were skipped:")
    for error in errors[:10]:
        print("-", error)


In [ ]:
# Cell 5: Retrieval and prompt construction
SYSTEM_PROMPT = (
    "You are CarbonTatvaAI, an enterprise ESG assistant. Answer using only the supplied context. "
    "If evidence is missing, say what is missing. Keep answers concise, structured, and suitable for ESG teams. "
    "Do not invent numbers, policies, peers, targets, certifications, or compliance claims."
)

def retrieve(question, top_k=TOP_K):
    query = vectorizer.transform([question])
    scores = cosine_similarity(query, matrix)[0]
    ranked = scores.argsort()[::-1][:top_k]
    results = []
    for index in ranked:
        if scores[index] <= 0 and len(results) > 0:
            continue
        results.append({
            "source": chunks[int(index)]["source"],
            "text": chunks[int(index)]["text"],
            "score": float(scores[index]),
        })
    return results

def build_prompt(question, retrieved):
    context = "\n\n".join(
        f"[{idx}] Source: {item['source']}\n{item['text']}"
        for idx, item in enumerate(retrieved, 1)
    )
    return (
        "Context excerpts:\n"
        f"{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer requirements:\n"
        "- Use only the context excerpts.\n"
        "- Cite source numbers like [1], [2] where useful.\n"
        "- If drafting a disclosure, write polished disclosure-ready text.\n"
        "- If comparing companies, separate the comparison points clearly.\n"
        "- Mention missing evidence instead of guessing."
    )


In [ ]:
# Cell 6: Ollama call with mock fallback
def call_ollama(prompt, model=OLLAMA_MODEL, url=OLLAMA_URL, timeout=180):
    body = json.dumps({
        "model": model,
        "system": SYSTEM_PROMPT,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.2,
            "num_ctx": 4096,
            "repeat_penalty": 1.05,
        },
    }).encode("utf-8")
    request = urllib.request.Request(
        url,
        data=body,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        payload = json.loads(response.read().decode("utf-8"))
    return clean_text(payload.get("response", ""))

def mock_answer(question, retrieved):
    # TODO: Replace this fallback by a deployed backend call when the public Ollama endpoint is ready.
    source_list = ", ".join(f"[{idx}] {item['source']}" for idx, item in enumerate(retrieved, 1))
    evidence_preview = " ".join(item["text"] for item in retrieved[:2])
    evidence_preview = textwrap.shorten(evidence_preview, width=700, placeholder="...")
    q = question.lower()
    if "board" in q and "oversight" in q:
        answer = (
            "Draft board oversight disclosure: The Board should disclose how ESG and climate-related matters are reviewed, "
            "which board committee or governance body is responsible, how often updates are received, and how management "
            "escalates climate risks and compliance issues. Based on the available context, any missing committee name, "
            "meeting frequency, or executive owner should be marked as not disclosed rather than assumed."
        )
    elif "brsr" in q and "energy" in q:
        answer = (
            "BRSR energy response: The available context indicates that energy reporting should cover total energy consumption, "
            "energy intensity, renewable and non-renewable energy where available, and year-on-year movement. If the input only "
            "contains KPI values, the answer should report those values and avoid unsupported claims about initiatives or targets."
        )
    elif "scope 3" in q:
        answer = (
            "Peer Scope 3 response: Peers usually disclose the relevant Scope 3 categories, calculation boundary, methodology, "
            "emission factors, data-quality limitations, and exclusions. A strong answer should identify which categories are "
            "available in evidence and clearly state missing categories."
        )
    elif "cbam" in q:
        answer = (
            "CBAM-aligned draft: The disclosure should describe product boundary, embedded emissions, methodology, verification "
            "status, and data gaps. It should avoid claiming CBAM compliance unless the evidence confirms verified CBAM reporting."
        )
    else:
        answer = (
            "Enterprise ESG answer: The retrieved context should be used to produce a grounded comparison or disclosure. "
            "The response should separate disclosed facts from missing evidence and should not invent company-specific claims."
        )
    return f"[MOCK MODE - Ollama not reachable]\n\n{answer}\n\nRetrieved evidence used: {source_list}\n\nEvidence preview: {evidence_preview}"

def ask_esg(question, top_k=TOP_K, force_mock=False):
    retrieved = retrieve(question, top_k=top_k)
    prompt = build_prompt(question, retrieved)
    if force_mock:
        return {"mode": "mock", "answer": mock_answer(question, retrieved), "sources": retrieved}
    try:
        answer = call_ollama(prompt)
        mode = "ollama"
        if not answer:
            raise RuntimeError("Empty Ollama response")
    except Exception as exc:
        answer = mock_answer(question, retrieved) + f"\n\nBackend note: {exc}"
        mode = "mock"
    return {"mode": mode, "answer": answer, "sources": retrieved}


In [ ]:
# Cell 7: Run the enterprise chatbot examples
EXAMPLE_QUESTIONS = [
    "How do peers disclose Scope 3?",
    "Generate a board oversight disclosure.",
    "What does BRSR require for energy?",
    "Compare us against ITC and Tata Steel.",
    "Draft CBAM-aligned disclosures.",
]

example_outputs = []
for question in EXAMPLE_QUESTIONS:
    result = ask_esg(question)
    example_outputs.append({"question": question, **result})
    print("=" * 100)
    print("Question:", question)
    print("Mode:", result["mode"])
    print(result["answer"])
    print("\nSources:")
    for idx, source in enumerate(result["sources"], 1):
        print(f"  [{idx}] {source['source']} score={source['score']:.3f}")
    print()

(WORK_DIR / "enterprise_chatbot_example_outputs.json").write_text(
    json.dumps(example_outputs, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Saved:", WORK_DIR / "enterprise_chatbot_example_outputs.json")


In [ ]:
# Cell 8: Ask your own question
MY_QUESTION = "Generate a KPI-grounded ESG summary for the uploaded company."

result = ask_esg(MY_QUESTION)
print("Question:", MY_QUESTION)
print("Mode:", result["mode"])
print("\nAnswer:\n", result["answer"])
print("\nSources:")
for idx, source in enumerate(result["sources"], 1):
    print(f"[{idx}] {source['source']} score={source['score']:.3f}")


In [ ]:
# Cell 9: Stitch / frontend integration contract
api_contract = {
    "name": "CarbonTatvaAI Enterprise ESG Chatbot",
    "frontend_input": {
        "question": "How do peers disclose Scope 3?",
        "optional_company_context": "Company name, sector, KPIs, peer names, or reporting standard",
    },
    "backend_endpoint": "POST /api/chat",
    "backend_request": {
        "question": "string",
        "top_k": 5,
    },
    "backend_response": {
        "mode": "ollama | mock",
        "answer": "string",
        "sources": [{"source": "file/page/table", "score": 0.0}],
    },
    "todo": "Connect Stitch UI to a small backend that calls ask_esg() or the deployed Ollama API.",
}

contract_path = WORK_DIR / "stitch_frontend_api_contract.json"
contract_path.write_text(json.dumps(api_contract, indent=2), encoding="utf-8")
print(json.dumps(api_contract, indent=2))
print("Saved:", contract_path)


## What to say in the demo

> I built an enterprise ESG chatbot notebook. It ingests ESG reports, BRSR/KPI tables, or extracted ESG text, retrieves relevant evidence, and answers questions such as Scope 3 peer disclosure, board oversight disclosure, BRSR energy requirements, peer comparison, and CBAM-aligned drafting. It calls our Ollama model when available and uses a marked mock fallback only when the model endpoint is not reachable.

For the Stitch UI, the frontend should call a backend endpoint with the user's question and display the answer plus retrieved sources.
